# Rebuild and publish the ShellAI GGUF

This converts `micrictor/shellai-qwen3-0.6b-qwen38-nl2bash-distilled` with current llama.cpp, validates that the expanded Qwen3.8 vocabulary, tensors, and ShellAI chat template survived conversion, compares greedy HF/F16/Q8 outputs, and only then uploads Q8_0 and Q4_K_M artifacts. Use a fresh GPU Colab runtime and add a write-capable `HF_TOKEN` secret. `HF_GGUF_REPO_ID` is optional.

In [ ]:
import os, subprocess, sys

def run_live(command, cwd=None, env=None):
    print('+', ' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(
        list(map(str, command)), cwd=cwd, env=env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
    )
    lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command, ''.join(lines))
    return ''.join(lines)

run_live(['nvidia-smi'])
run_live([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
          'transformers>=5,<6', 'huggingface-hub>=0.34,<2',
          'safetensors>=0.6,<1', 'accelerate>=1.10,<2', 'sentencepiece'])

In [ ]:
import json
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi, snapshot_download

SOURCE_REPO = 'micrictor/shellai-qwen3-0.6b-qwen38-nl2bash-distilled'
SOURCE_DIR = Path('/content/shellai-qwen3-hf')
OUTPUT_DIR = Path('/content/shellai-qwen3-gguf')
LLAMA_DIR = Path('/content/llama.cpp')
OUTPUT_DIR.mkdir(exist_ok=True)

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception as error:
    raise RuntimeError('Add a write-capable HF_TOKEN in Colab Secrets.') from error
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is empty.')
api = HfApi(token=HF_TOKEN)
account = api.whoami()['name']
try:
    secret_target = userdata.get('HF_GGUF_REPO_ID')
except Exception:
    secret_target = None
TARGET_REPO = secret_target or f'{account}/Qwen3-0.6B-Qwen38-ShellAI-GGUF'
print('Source:', SOURCE_REPO)
print('Target:', TARGET_REPO)
snapshot_download(SOURCE_REPO, local_dir=SOURCE_DIR, token=HF_TOKEN)

# llama.cpp currently pins Transformers 4.57.6, which predates the named
# Qwen3_5Tokenizer class. The complete tokenizer implementation is already
# serialized in tokenizer.json, so use the generic fast-tokenizer loader in
# this local conversion copy. The Hub source repository is not modified.
tokenizer_json = SOURCE_DIR / 'tokenizer.json'
tokenizer_config_path = SOURCE_DIR / 'tokenizer_config.json'
if not tokenizer_json.is_file():
    raise FileNotFoundError(f'Missing tokenizer artifact: {tokenizer_json}')
tokenizer_config = json.loads(tokenizer_config_path.read_text(encoding='utf-8'))
original_tokenizer_class = tokenizer_config.get('tokenizer_class')
if original_tokenizer_class == 'Qwen3_5Tokenizer':
    tokenizer_config['tokenizer_class'] = 'PreTrainedTokenizerFast'
    tokenizer_config_path.write_text(
        json.dumps(tokenizer_config, indent=2, sort_keys=True) + '\n', encoding='utf-8'
    )
print('Conversion tokenizer class:', original_tokenizer_class, '->', tokenizer_config['tokenizer_class'])

## Validate the actual source checkpoint

These checks catch the exact failure in the old GGUF before conversion starts.

In [ ]:
import math
from safetensors import safe_open
from transformers import AutoConfig, AutoTokenizer

config = AutoConfig.from_pretrained(SOURCE_DIR, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(SOURCE_DIR, local_files_only=True)
messages = [
    {'role': 'system', 'content': 'You are a helpful assistant that translates natural language to bash commands.'},
    {'role': 'user', 'content': 'Generate single Bash command: list every file modified today'},
]
rendered_prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
expected_suffix = (
    '<|im_start|>user\nGenerate one Bash command: list every file modified today<|im_end|>\n'
    '<|im_start|>assistant\n<think>\n\n</think>\n\n'
)
assert config.vocab_size == 248320, config.vocab_size
assert rendered_prompt.endswith(expected_suffix), rendered_prompt
weight_count = 0
for weight_file in SOURCE_DIR.glob('*.safetensors'):
    with safe_open(weight_file, framework='pt', device='cpu') as weights:
        for name in weights.keys():
            weight_count += math.prod(weights.get_slice(name).get_shape())
assert weight_count >= 650_000_000, weight_count
print('config vocab:', config.vocab_size)
print('tokenizer entries:', len(tokenizer))
print('stored tensor elements:', f'{weight_count:,}')
print('Rendered prompt:\n' + rendered_prompt)

In [ ]:
import gc, torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    SOURCE_DIR, dtype=torch.bfloat16, device_map={'': 0}, local_files_only=True
)
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True,
    return_dict=True, return_tensors='pt',
).to('cuda')
with torch.inference_mode():
    generated = model.generate(**inputs, max_new_tokens=128, do_sample=False)
HF_PREDICTION = tokenizer.decode(
    generated[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True
).strip()
assert HF_PREDICTION, 'Transformers checkpoint generated an empty response'
print('HF:', HF_PREDICTION)
del model, inputs, generated
gc.collect(); torch.cuda.empty_cache()

## Build current llama.cpp and convert

The full F16 conversion is retained until parity is proven. Conversion and build output stream live in the notebook.

In [ ]:
if not LLAMA_DIR.exists():
    run_live(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', str(LLAMA_DIR)])
else:
    # Colab can preserve /content across notebook reconnects. An older
    # llama.cpp Qwen3 converter required tokenizer.model, but this distilled
    # checkpoint intentionally carries the Qwen3.8 tokenizer.json vocabulary.
    run_live(['git', '-C', str(LLAMA_DIR), 'pull', '--ff-only'])
qwen_converter = (LLAMA_DIR / 'conversion/qwen.py').read_text(encoding='utf-8')
if 'except FileNotFoundError:' not in qwen_converter or '_set_vocab_gpt2()' not in qwen_converter:
    raise RuntimeError('llama.cpp is too old to convert a Qwen3 body with a tokenizer.json vocabulary')
run_live([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(LLAMA_DIR / 'requirements.txt')])
tokenizer_smoke = (
    "import sys, transformers; from transformers import AutoTokenizer; "
    "t=AutoTokenizer.from_pretrained(sys.argv[1], local_files_only=True); "
    "print('converter transformers', transformers.__version__, '| tokenizer', type(t).__name__, '| entries', len(t))"
)
run_live([sys.executable, '-c', tokenizer_smoke, SOURCE_DIR])
run_live(['cmake', '-S', str(LLAMA_DIR), '-B', str(LLAMA_DIR / 'build'),
          '-DGGML_CUDA=ON', '-DLLAMA_CURL=OFF', '-DCMAKE_BUILD_TYPE=Release'])
run_live(['cmake', '--build', str(LLAMA_DIR / 'build'), '--config', 'Release',
          '-j', '2', '--target', 'llama-cli', 'llama-quantize'])

F16 = OUTPUT_DIR / 'Qwen3-0.6B-Qwen38-ShellAI-F16.gguf'
Q8 = OUTPUT_DIR / 'Qwen3-0.6B-Qwen38-ShellAI-Q8_0.gguf'
Q4 = OUTPUT_DIR / 'Qwen3-0.6B-Qwen38-ShellAI-Q4_K_M.gguf'
run_live([sys.executable, LLAMA_DIR / 'convert_hf_to_gguf.py', SOURCE_DIR,
          '--outfile', F16, '--outtype', 'f16'])
QUANTIZE = LLAMA_DIR / 'build/bin/llama-quantize'
run_live([QUANTIZE, F16, Q8, 'Q8_0'])
run_live([QUANTIZE, F16, Q4, 'Q4_K_M'])

## Inspect every GGUF structurally

A file with an unexpanded vocabulary, old template, or undersized tensor set fails here.

In [ ]:
import numpy as np
sys.path.insert(0, str(LLAMA_DIR / 'gguf-py'))
from gguf import GGUFReader

def field_scalar(reader, name):
    field = reader.get_field(name)
    if field is None or len(field.data) != 1:
        raise AssertionError(f'missing scalar GGUF field: {name}')
    value = field.parts[field.data[0]]
    if value.dtype == np.uint8:
        return bytes(value).decode('utf-8')
    return value.reshape(-1)[0].item()

def validate_gguf(path):
    reader = GGUFReader(path)
    architecture = field_scalar(reader, 'general.architecture')
    vocab_size = field_scalar(reader, 'qwen3.vocab_size')
    template = field_scalar(reader, 'tokenizer.chat_template')
    token_count = len(reader.get_field('tokenizer.ggml.tokens').data)
    tensor_elements = sum(math.prod(tensor.shape) for tensor in reader.tensors)
    embeddings = next(t for t in reader.tensors if t.name == 'token_embd.weight')
    assert architecture == 'qwen3', architecture
    assert vocab_size == 248320, vocab_size
    assert token_count >= len(tokenizer), token_count
    assert 248320 in embeddings.shape, embeddings.shape
    assert tensor_elements >= 650_000_000, tensor_elements
    assert 'Generate single Bash command: ' in template
    assert 'Generate one Bash command: ' in template
    assert '<think>\n\n</think>' in template
    print(path.name, '| vocab', vocab_size, '| tokens', token_count,
          '| elements', f'{tensor_elements:,}', '| embedding', tuple(embeddings.shape))

for artifact in (F16, Q8, Q4):
    validate_gguf(artifact)

## Require known-answer parity before upload

Both F16 and Q8 must reproduce the direct Transformers answer. Q4 is printed for inspection but does not gate the Q8 publication.

In [ ]:
LLAMA_CLI = LLAMA_DIR / 'build/bin/llama-cli'

def llama_prediction(path):
    command = [
        str(LLAMA_CLI), '-m', str(path), '--prompt', rendered_prompt,
        '--n-predict', '128', '--temp', '0', '--no-display-prompt',
        '--no-warmup', '--no-show-timings', '--simple-io', '-ngl', '999',
    ]
    result = subprocess.run(command, text=True, capture_output=True)
    print(result.stderr, end='', flush=True)
    print(result.stdout, end='', flush=True)
    if result.returncode:
        raise subprocess.CalledProcessError(result.returncode, command, result.stdout, result.stderr)
    return result.stdout.strip()

predictions = {artifact.name: llama_prediction(artifact) for artifact in (F16, Q8, Q4)}
print('\\nHF:', HF_PREDICTION)
for name, prediction in predictions.items():
    print(name + ':', prediction)
assert predictions[F16.name] == HF_PREDICTION, predictions[F16.name]
assert predictions[Q8.name] == HF_PREDICTION, predictions[Q8.name]

## Upload the validated replacements

Run this cell only after reviewing the comparison above. It replaces the existing filenames with validated files in a new Hub commit.

In [ ]:
api.create_repo(TARGET_REPO, repo_type='model', private=False, exist_ok=True)
for artifact in (Q8, Q4):
    print('Uploading', artifact.name, flush=True)
    commit = api.upload_file(
        path_or_fileobj=artifact, path_in_repo=artifact.name, repo_id=TARGET_REPO,
        repo_type='model', commit_message=f'Rebuild {artifact.name} from {SOURCE_REPO}',
    )
    print(commit)
remote_files = set(api.list_repo_files(TARGET_REPO, repo_type='model'))
required = {Q8.name, Q4.name}
assert required <= remote_files, required - remote_files
print('Validated GGUF repository: https://huggingface.co/' + TARGET_REPO)